In [2]:
import time
import random
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

I0000 00:00:1792701770.670502  161378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792701770.956190  161378 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792701772.300306  161378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cuda


# 1. Same network as in the PyTorch CNN tutorial

In [3]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')


correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')


# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

[1,  2000] loss: 2.185
[1,  4000] loss: 1.933
[1,  6000] loss: 1.687
[1,  8000] loss: 1.583
[1, 10000] loss: 1.533
[1, 12000] loss: 1.477
[2,  2000] loss: 1.378
[2,  4000] loss: 1.374
[2,  6000] loss: 1.374
[2,  8000] loss: 1.336
[2, 10000] loss: 1.306
[2, 12000] loss: 1.280
Finished Training
Accuracy of the network on the 10000 test images: 48 %
Accuracy for class: plane is 60.5 %
Accuracy for class: car   is 22.6 %
Accuracy for class: bird  is 39.1 %
Accuracy for class: cat   is 28.3 %
Accuracy for class: deer  is 26.0 %
Accuracy for class: dog   is 43.6 %
Accuracy for class: frog  is 64.2 %
Accuracy for class: horse is 62.3 %
Accuracy for class: ship  is 77.6 %
Accuracy for class: truck is 60.3 %


# 2. Architecture change

In [4]:
# Tieferes Netz: 4 Conv-Schichten mit 3x3-Kernen statt 2 mit 5x5
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # padding=1 erhaelt die raeumliche Groesse (32x32) bei 3x3-Kernen
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)    # 3 -> 8 Kanaele
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)   # 8 -> 16 Kanaele
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)  # 16 -> 32 Kanaele
        self.conv4 = nn.Conv2d(32, 64, 3, padding=1)  # 32 -> 64 Kanaele
        self.pool = nn.MaxPool2d(2, 2)                # halbiert Hoehe und Breite
        # nach zweimal Pooling: 32x32 -> 16x16 -> 8x8, also 64 * 8 * 8 Eingaenge
        self.fc1 = nn.Linear(64 * 8 * 8, 4096)
        self.fc2 = nn.Linear(4096, 1000)
        self.fc3 = nn.Linear(1000, 10)                # 10 Ausgaben = 10 CIFAR-Klassen

    def forward(self, x):
        x = F.relu(self.conv1(x))                     # 8 x 32 x 32
        x = self.pool(F.relu(self.conv2(x)))          # 16 x 16 x 16
        x = F.relu(self.conv3(x))                     # 32 x 16 x 16
        x = self.pool(F.relu(self.conv4(x)))          # 64 x 8 x 8
        x = torch.flatten(x, 1)                       # alle Dimensionen ausser Batch flach machen
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)                            # rohe Logits, Softmax steckt im CrossEntropyLoss

## Training and evaluation helpers
`train()` logs the training loss to TensorBoard (see section 4), `evaluate()` reports top-1 and top-3 accuracy (see section 5).

In [ ]:
def train(net, device, writer, epochs=2, lr=0.001, loader=None, log_every=2000):
    """Trainiert das Netz auf dem angegebenen Device, loggt den Loss nach
    TensorBoard und gibt die Trainingsdauer in Sekunden zurueck."""
    loader = loader or trainloader
    net.to(device)   # Gewichte auf CPU bzw. GPU verschieben
    net.train()      # Trainingsmodus (relevant z.B. fuer Dropout/BatchNorm)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    start = time.perf_counter()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(loader, 0):
            # Daten muessen auf demselben Device liegen wie das Netz
            inputs, labels = inputs.to(device), labels.to(device)

            # Gradienten zuruecksetzen, Forward-Pass, Backpropagation, Gewichtsupdate
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # alle log_every Mini-Batches den mittleren Loss ausgeben und loggen
            running_loss += loss.item()
            if i % log_every == log_every - 1:
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / log_every:.3f}')
                # x-Achse in TensorBoard = Anzahl bisher gesehener Bilder
                writer.add_scalar('training loss', running_loss / log_every,
                                  (epoch * len(loader) + i) * loader.batch_size)
                running_loss = 0.0
    # CUDA arbeitet asynchron: erst auf alle Kernels warten, sonst ist die Zeitmessung zu kurz
    if device.type == 'cuda':
        torch.cuda.synchronize()
    duration = time.perf_counter() - start
    writer.flush()   # gepufferte Eintraege auf die Platte schreiben
    print(f'Finished Training on {device} in {duration:.1f} s')
    return duration

## Training on the CPU (baseline for the speedup comparison)

In [6]:
# fester Seed, damit CPU- und GPU-Netz mit identischen Startgewichten beginnen
torch.manual_seed(0)
net_cpu = Net()

# passiv aufgabe 4: eigenes TensorBoard-Log fuer den CPU-Lauf
cpu_writer = SummaryWriter('runs/cifar10/net_cpu')
# Netzgraph loggen; images ist ein Beispiel-Batch aus der Test-Schleife in Abschnitt 1
cpu_writer.add_graph(net_cpu, images)
cpu_time = train(net_cpu, torch.device('cpu'), cpu_writer)
cpu_writer.close()

/home/debian/jupyter-env/lib/python3.14/site-packages/torch/jit/_trace.py:1000: FutureWarning: `torch.jit.trace` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/debian/jupyter-env/lib/python3.14/site-packages/torch/jit/_trace.py:1139: FutureWarning: `torch.jit.trace_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[1,  2000] loss: 2.286
[1,  4000] loss: 1.959
[1,  6000] loss: 1.750
[1,  8000] loss: 1.636
[1, 10000] loss: 1.514
[1, 12000] loss: 1.441
[2,  2000] loss: 1.302
[2,  4000] loss: 1.242
[2,  6000] loss: 1.200
[2,  8000] loss: 1.141
[2, 10000] loss: 1.106
[2, 12000] loss: 1.053
Finished Training on cpu in 1255.7 s


# 3. Train on GPU

In [7]:
# gleicher Seed wie beim CPU-Lauf -> gleiche Startgewichte, fairer Vergleich
torch.manual_seed(0)
net = Net()

# passiv aufgabe 4: eigenes TensorBoard-Log fuer den GPU-Lauf
gpu_writer = SummaryWriter('runs/cifar10/net_gpu')
gpu_writer.add_graph(net, images)  # add_graph laeuft vor dem Verschieben, Netz und images liegen noch auf der CPU
gpu_time = train(net, device, gpu_writer)
# gpu_writer bleibt offen, er wird in Abschnitt 6 noch fuer die Top-3-Fehler genutzt

# Laufzeiten vergleichen
print(f'CPU: {cpu_time:.1f} s')
print(f'GPU: {gpu_time:.1f} s')
print(f'Speedup: {cpu_time / gpu_time:.2f}x')

[1,  2000] loss: 2.287
[1,  4000] loss: 1.960
[1,  6000] loss: 1.752
[1,  8000] loss: 1.637
[1, 10000] loss: 1.514
[1, 12000] loss: 1.443
[2,  2000] loss: 1.304
[2,  4000] loss: 1.244
[2,  6000] loss: 1.201
[2,  8000] loss: 1.142
[2, 10000] loss: 1.111
[2, 12000] loss: 1.049
Finished Training on cuda in 105.7 s
CPU: 1255.7 s
GPU: 105.7 s
Speedup: 11.88x


# 5. Top 3 accuracy

In [10]:
def evaluate(net, device, k=3):
    """Berechnet Top-1- und Top-k-Accuracy auf dem Testset und gibt die Top-k-Accuracy zurueck."""
    net.to(device)
    net.eval()   # Evaluationsmodus
    top1 = topk = total = 0
    with torch.no_grad():   # keine Gradienten noetig -> schneller, weniger Speicher
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            # Indizes der k Klassen mit den hoechsten Scores, Form: (batch, k), absteigend sortiert
            top = outputs.topk(k, dim=1).indices
            # Top-1: die beste Vorhersage stimmt mit dem Label ueberein
            top1 += (top[:, 0] == labels).sum().item()
            # Top-k: das Label kommt irgendwo unter den k besten Vorhersagen vor
            topk += (top == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)
    print(f'Top-1 accuracy: {100 * top1 / total:.2f} %')
    print(f'Top-{k} accuracy: {100 * topk / total:.2f} %')
    return topk / total

evaluate(net, device)

Top-1 accuracy: 63.98 %
Top-3 accuracy: 89.62 %


0.8962

# 6. Top-3 errors in TensorBoard
Random test images whose true label is not among the top three predictions, written to TensorBoard.

In [11]:
def top3_errors(net, device, n=5):
    """Sammelt alle Testbilder, deren wahres Label nicht unter den Top-3
    liegt, und gibt n zufaellig gewaehlte davon zurueck."""
    net.to(device)
    net.eval()
    errors = []
    with torch.no_grad():
        for images, labels in testloader:
            # Ausgaben zurueck auf die CPU, damit sie mit den CPU-Bildern verglichen werden koennen
            outputs = net(images.to(device)).cpu()
            top = outputs.topk(3, dim=1).indices
            for img, label, pred in zip(images, labels, top):
                if label not in pred:   # wahres Label fehlt in den Top-3 -> Fehler
                    errors.append((img, label.item(), pred.tolist()))
    return random.sample(errors, n)


samples = top3_errors(net, device)

# Matplotlib-Figure mit den 5 Fehlbildern, Titel zeigt wahres Label und Top-3-Vorhersage
fig = plt.figure(figsize=(15, 4))
for idx, (img, label, pred) in enumerate(samples):
    ax = fig.add_subplot(1, 5, idx + 1, xticks=[], yticks=[])
    # Normalisierung rueckgaengig machen ([-1, 1] -> [0, 1]) und CHW -> HWC fuer imshow
    ax.imshow(np.transpose((img / 2 + 0.5).numpy(), (1, 2, 0)))
    ax.set_title(f'true: {classes[label]}\npred: {", ".join(classes[p] for p in pred)}')

# Fehler auf drei Arten nach TensorBoard schreiben: als Figure, als Bildraster und als Text
gpu_writer.add_figure('top-3 errors', fig)
gpu_writer.add_images('top-3 error images', torch.stack([s[0] for s in samples]) / 2 + 0.5)
gpu_writer.add_text('top-3 error labels', '  \n'.join(
    f'{i}: true={classes[l]}, pred={[classes[p] for p in pr]}' for i, (_, l, pr) in enumerate(samples)))
gpu_writer.close()

## Precision-recall curves per class

In [ ]:
# Softmax-Wahrscheinlichkeiten und wahre Labels fuer das gesamte Testset sammeln
probs, targets = [], []
net.eval()
with torch.no_grad():
    for images, labels in testloader:
        probs.append(F.softmax(net(images.to(device)), dim=1).cpu())
        targets.append(labels)
# Listen von Batches zu einem Tensor zusammenfuegen: probs (10000, 10), targets (10000,)
probs, targets = torch.cat(probs), torch.cat(targets)

# eine PR-Kurve pro Klasse (One-vs-Rest): positiv = Bild gehoert zu Klasse i,
# Score = vorhergesagte Wahrscheinlichkeit fuer Klasse i
pr_writer = SummaryWriter('runs/cifar10/net_gpu')
for i, classname in enumerate(classes):
    pr_writer.add_pr_curve(classname, targets == i, probs[:, i], global_step=0)
pr_writer.close()


# Interpretation der Ergebnisse und Begründungen

Aufbau pro Abschnitt: **Ergebnis** → **Warum so umgesetzt?** → **Mögliche Fragen** mit Antwort.

---

## 1. Netz aus dem PyTorch-Tutorial

**Ergebnis**
- Loss sinkt von 2,19 auf 1,28, Test-Accuracy **48 %** (Raten: 10 %).
- Klassen-Accuracies streuen stark (car 22,6 % bis ship 77,6 %).
- Das Netz ist klein (**ca. 62k Parameter**) und wurde nur 2 Epochen trainiert. Der Loss fällt am Ende noch, das Netz ist also noch nicht konvergiert (eher Underfitting als Overfitting).

**Warum so umgesetzt?**
- `ToImage` + `ToDtype(scale=True)` bringt die Pixel auf [0, 1], `Normalize(0.5, 0.5)` dann auf [-1, 1]. Zentrierte Eingaben um 0 führen zu stabileren Gradienten.
- `shuffle=True` beim Training, damit die Mini-Batches in jeder Epoche anders gemischt sind (sonst lernt das Netz Reihenfolge-Effekte). Beim Testen `shuffle=False`, weil die Reihenfolge für die Accuracy egal ist.
- `num_workers=2`: Daten werden in separaten Prozessen geladen, während das Netz rechnet.

**Mögliche Fragen**
- *Warum steht im `forward` kein Softmax?*
  `nn.CrossEntropyLoss` erwartet rohe Logits und rechnet intern `log_softmax` + Negative Log-Likelihood, numerisch stabil in einem Schritt. Ein zusätzliches Softmax würde doppelt normalisieren und die Gradienten verschlechtern. Softmax braucht man nur, wenn man echte Wahrscheinlichkeiten sehen will (z.B. bei den PR-Kurven).
- *Woher kommt `16 * 5 * 5` in `fc1`?*
  32×32 → conv 5×5 (ohne Padding) → 28×28 → pool → 14×14 → conv 5×5 → 10×10 → pool → 5×5, mit 16 Kanälen. `flatten` macht daraus (N, 16·5·5) = (N, 400).
- *Warum `optimizer.zero_grad()`?*
  PyTorch summiert Gradienten bei jedem `backward()` auf. Ohne Zurücksetzen würden sich die Gradienten aller bisherigen Batches addieren.
- *Was macht `momentum=0.9`?*
  Das Update berücksichtigt den gleitenden Durchschnitt der bisherigen Gradienten. Dadurch wird das Training in konsistente Richtungen schneller und schwankt weniger.
- *Warum `torch.no_grad()` beim Testen?*
  Für die Auswertung braucht man keine Gradienten. Ohne Rechengraph spart man Speicher und Zeit.
- *Warum sind die Klassen-Accuracies so unterschiedlich?*
  Nach 2 Epochen ist das Netz unfertig und hat noch eine Vorliebe für bestimmte Klassen. Ein Durchlauf ohne festen Seed liefert deshalb stark schwankende Werte pro Klasse (ein zweiter Lauf ergab z.B. 56 % gesamt und car 75 %).

---

## 2. Geänderte Architektur

**Ergebnis**
- **Top-1: 63,98 % statt 48 %** (+16 Prozentpunkte) bei gleicher Epochenzahl, Loss am Ende 1,05 statt 1,28.

**Architektur laut Aufgabe** (conv3×3: 8 → 16 → pool → 32 → 64 → pool → FC 4096 → 1000 → 10):

| Schicht | Ausgabeform | Parameter |
|---|---|---|
| conv1 3→8, 3×3 | 8×32×32 | 224 |
| conv2 8→16, 3×3 + pool | 16×16×16 | 1 168 |
| conv3 16→32, 3×3 | 32×16×16 | 4 640 |
| conv4 32→64, 3×3 + pool | 64×8×8 | 18 496 |
| fc1 4096→4096 | 4096 | 16 781 312 |
| fc2 4096→1000 | 1000 | 4 097 000 |
| fc3 1000→10 | 10 | 10 010 |
| **Summe** | | **≈ 20,9 Mio.** |

**Warum so umgesetzt?**
- **`padding=1` ist unsere Entscheidung**, die Aufgabe gibt es nicht vor. Mit Padding bleibt die Bildgröße pro Conv erhalten, nur das Pooling halbiert: 32 → 16 → 8. Ohne Padding wäre es 32 → 30 → 28 → 14 → 12 → 10 → 5, und Informationen am Bildrand gingen bei jedem Conv verloren. Bei nur 32×32 Pixeln ist das ein relevanter Anteil.
- Die Eingangsgröße von `fc1` ergibt sich daraus: 64 Kanäle × 8 × 8 = 4096.
- Logits am Ende, Softmax steckt wie oben im Loss.

**Mögliche Fragen**
- *Warum ist das neue Netz besser?*
  Mehr Conv-Schichten bedeuten eine tiefere Merkmalshierarchie (Kanten → Texturen → Formen). Mehr Kanäle (bis 64) erlauben mehr unterschiedliche Filter. Durch das Padding bleibt mehr räumliche Information erhalten (8×8 statt 5×5). Außerdem hat das Netz insgesamt viel mehr Kapazität. Welcher Faktor wie viel ausmacht, haben wir nicht getrennt untersucht (dafür bräuchte man eine Ablationsstudie).
- *Warum 3×3-Kernel statt 5×5?*
  Zwei 3×3-Convs hintereinander sehen denselben Bildausschnitt wie ein 5×5-Conv (rezeptives Feld 5×5). Sie haben aber weniger Parameter (2·9 = 18 statt 25 pro Kanalpaar) und eine zusätzliche ReLU dazwischen, also mehr Nichtlinearität. Das ist die Idee hinter VGG.
- *Wo stecken die Parameter?*
  Über 99 % in den FC-Schichten, allein `fc1` hat 16,8 Mio. Die Conv-Schichten haben zusammen nur ca. 24k. Conv-Schichten teilen ihre Gewichte über das ganze Bild (Weight Sharing), FC-Schichten verbinden jeden Eingang mit jedem Ausgang.
- *Overfitting?*
  Bei 20,9 Mio. Parametern und 50 000 Trainingsbildern ist das Risiko bei längerem Training hoch. Nach 2 Epochen sinkt der Loss aber noch, und wir haben keine Trainings-Accuracy gemessen. Belegen können wir Overfitting also nicht. Prüfen könnte man es, indem man Trainings- und Test-Accuracy über die Epochen vergleicht.
- *Was könnte man verbessern (Bonus)?*
  BatchNorm, Dropout in den FC-Schichten, Data Augmentation (Random Crop, Horizontal Flip), mehr Epochen mit Learning-Rate-Schedule, größere Batches oder Global Average Pooling statt riesiger FC-Schichten. Mit `nn.Sequential` ließe sich die Architektur kompakter schreiben.

---

## 3. Training auf der GPU vs. CPU

**Ergebnis**
- CPU: 1255,7 s, GPU (RTX 5060 Ti): 105,7 s → **Speedup 11,9×**.
- Die Loss-Verläufe sind fast identisch (z.B. 1,053 vs. 1,049 am Ende).

**Warum so umgesetzt?**
- **`torch.manual_seed(0)` vor beiden Läufen**: Beide Netze starten mit denselben Gewichten. Da der DataLoader beim Mischen ebenfalls den globalen Zufallsgenerator nutzt, sehen beide auch dieselbe Batch-Reihenfolge. Der Vergleich misst dann nur die Hardware und nicht den Zufall.
- **`net.to(device)` und `inputs.to(device)`**: Netz und Daten müssen auf demselben Gerät liegen, sonst gibt es einen Fehler.
- **`torch.cuda.synchronize()` vor dem Stoppen der Zeit**: CUDA-Aufrufe sind asynchron. Python läuft weiter, während die GPU noch rechnet. Ohne Synchronisieren wäre die gemessene Zeit zu kurz.
- **`time.perf_counter()`** ist die genaueste Uhr für Laufzeitmessungen in Python.
- Eine gemeinsame `train()`-Funktion für beide Läufe, damit sich nur das Gerät unterscheidet.

**Mögliche Fragen**
- *Warum ist der Loss nicht exakt gleich?*
  Gleitkomma-Addition ist nicht assoziativ. GPU und CPU summieren in anderer Reihenfolge und nutzen andere Algorithmen (cuDNN). Die kleinen Rundungsunterschiede summieren sich über 25 000 Updates.
- *Warum „nur“ 12× schneller?*
  Bei **Batchgröße 4** ist die GPU kaum ausgelastet. Pro Batch fallen feste Kosten an: Kernel-Starts, Kopieren der Daten von CPU zu GPU, Python-Overhead und das Laden der Daten im DataLoader. Bei 12 500 Batches pro Epoche dominieren diese Kosten. Mit Batchgröße 64–256 wäre der Speedup deutlich größer.
- *Warum ist die CPU so langsam?*
  Vor allem wegen der riesigen FC-Schichten: `fc1` braucht pro Bild ca. 16,8 Mio. Multiplikationen, die eine GPU massiv parallel ausführt.
- *Was bedeutet die Meldung „Could not find cuda drivers“?*
  Sie kommt von TensorFlow, das beim Import von TensorBoard mitgeladen wird, nicht von PyTorch. PyTorch nutzt die GPU trotzdem (Ausgabe `cuda`).

---

## 4. TensorBoard (Loss und Netzgraph)

**Warum so umgesetzt?**
- Getrennte Log-Ordner `runs/cifar10/net_cpu` und `net_gpu`, damit TensorBoard beide Läufe als eigene Kurven anzeigt und man sie übereinanderlegen kann.
- Der Loss wird alle 2000 Batches als Mittelwert geloggt, nicht pro Batch. Einzelne Batches mit 4 Bildern schwanken stark, der Mittelwert zeigt den Trend.
- x-Achse = Anzahl gesehener Bilder (`(epoch * len(loader) + i) * batch_size`). Diese Größe ist unabhängig von der Batchgröße vergleichbar.
- `add_graph(net, images)` braucht ein Beispiel-Input, weil PyTorch den Graphen erst durch einen Forward-Pass ermittelt (Tracing). Deshalb kommt die Warnung zu `torch.jit.trace` unter Python 3.14, die harmlos ist.
- `writer.flush()` bzw. `close()` schreiben gepufferte Einträge auf die Festplatte, sonst fehlen sie in TensorBoard.

---

## 5. Top-3-Accuracy

**Ergebnis**
- **Top-1: 63,98 %, Top-3: 89,62 %.** In ca. 26 % der Fälle liegt die richtige Klasse auf Platz 2 oder 3. Völlig daneben liegt das Netz nur bei ca. 10 % der Bilder.
- Zufälliges Raten ergäbe 10 % Top-1 und 30 % Top-3.

**Warum so umgesetzt?**
- `outputs.topk(3, dim=1).indices` liefert die Indizes der 3 höchsten Logits pro Bild, Form (Batch, 3).
- `(top == labels.unsqueeze(1)).any(dim=1)`: Das Label wird zu (Batch, 1) umgeformt und per Broadcasting mit allen 3 Vorhersagen verglichen. Ist eine davon richtig, zählt das Bild als Treffer. Das läuft vektorisiert ohne Python-Schleife über die Bilder.
- Top-1 und Top-3 werden im selben Durchlauf berechnet (`top[:, 0]` ist die beste Vorhersage).
- `net.eval()`: Für dieses Netz ohne Dropout/BatchNorm ändert es nichts. Es ist aber Standard und wichtig, sobald solche Schichten dazukommen.

**Mögliche Fragen**
- *Muss man für Top-k Softmax anwenden?*
  Nein. Softmax ist monoton und ändert die Reihenfolge der Logits nicht, `topk` auf Logits liefert dieselben Klassen.
- *Wozu ist Top-3 sinnvoll?*
  Top-3 zeigt, ob das Netz „ungefähr richtig“ liegt, also die richtige Klasse unter den Kandidaten hat. Das ist bei ähnlichen Klassen (Katze/Hund) aussagekräftiger als Top-1. Bei ImageNet mit 1000 Klassen ist Top-5 deshalb eine Standardmetrik.

---

## 6. Top-3-Fehler in TensorBoard

**Ergebnis (5 zufällige Bilder)**

| wahr | Top-3-Vorhersage |
|---|---|
| bird | horse, dog, cat |
| deer | cat, horse, frog |
| dog | ship, cat, bird |
| frog | bird, cat, deer |
| truck | horse, cat, dog |

- Fast alle Fehler sind **Tier ↔ Tier**. „cat“ steht in 4 von 5 Vorhersagen, das Netz nimmt „cat“ offenbar gern als Verlegenheitslösung für Tiere.
- Ausreißer wie dog → ship oder truck → horse liegen vermutlich an der niedrigen Auflösung, an untypischen Farben oder Hintergründen oder an ungewöhnlichen Bildausschnitten.
- 5 Bilder sind keine repräsentative Stichprobe. Bei jedem Lauf werden andere gezogen (`random.sample` ohne Seed).

**Warum so umgesetzt?**
- Zuerst werden **alle** Top-3-Fehler im Testset gesammelt und dann 5 zufällig gezogen. So ist jeder Fehler gleich wahrscheinlich. Nähme man einfach die ersten 5, kämen sie immer aus den ersten Test-Batches.
- Die Ausgaben werden mit `.cpu()` zurückgeholt, weil die Bilder für die Darstellung auf der CPU bleiben.
- `img / 2 + 0.5` macht die Normalisierung rückgängig ([-1, 1] → [0, 1]), sonst sähen die Bilder falsch aus. `np.transpose(..., (1, 2, 0))` stellt von PyTorch-Format C×H×W auf das von matplotlib erwartete H×W×C um.
- Drei Darstellungen in TensorBoard: als Figure (Bild mit Titel aus wahrem Label und Vorhersage), als Bildraster und als Text. Damit sind Bilder, wahre Labels und Vorhersagen wie gefordert geloggt.

---

## Zusatz: Precision-Recall-Kurven pro Klasse

**Ergebnis** (Fläche unter der PR-Kurve, aus den TensorBoard-Daten näherungsweise berechnet):

| Klasse | Fläche | | Klasse | Fläche |
|---|---|---|---|---|
| car | 0,85 | | horse | 0,75 |
| ship | 0,81 | | dog | 0,62 |
| plane | 0,79 | | deer | 0,62 |
| frog | 0,79 | | bird | 0,56 |
| truck | 0,76 | | **cat** | **0,48** |

- **Fahrzeuge erkennt das Netz besser als Tiere.** Sie haben klarere Formen und typische Hintergründe (Wasser, Himmel, Straße).
- **cat** ist die schwächste Klasse, gefolgt von bird, deer und dog. Das passt zu den Top-3-Fehlern: „cat“ wird oft fälschlich vorhergesagt, und das drückt die Precision.

**Warum so umgesetzt?**
- Hier ist Softmax nötig: `add_pr_curve` erwartet Werte zwischen 0 und 1, weil es Schwellwerte von 0 bis 1 durchläuft.
- One-vs-Rest pro Klasse: positiv = „Bild gehört zu Klasse i“, Score = Wahrscheinlichkeit für Klasse i.

**Mögliche Fragen**
- *Was zeigt eine PR-Kurve?*
  Für jeden Schwellwert: Precision (wie viele der als Klasse i vorhergesagten Bilder sind wirklich i) gegen Recall (wie viele der echten i-Bilder werden gefunden). Je weiter oben rechts die Kurve liegt, desto besser.
- *Warum nicht nur Accuracy?*
  Accuracy ist ein einzelner Wert für alle Klassen bei einer festen Entscheidung (argmax). PR-Kurven zeigen pro Klasse, wie gut das Netz Positive von Negativen trennt, unabhängig vom Schwellwert.

---

## Fazit
- Das tiefere Netz mit 3×3-Kerneln und Padding bringt schon nach 2 Epochen +16 Prozentpunkte Accuracy.
- Die GPU beschleunigt das Training um knapp das 12-Fache. Mit größeren Batches ginge deutlich mehr.
- Die verbleibenden Fehler betreffen vor allem ähnliche Tierklassen.

**Einschränkungen:** Jede Variante ist nur einmal gelaufen, und Abschnitt 1 hat keinen festen Seed. Die Zahlen schwanken deshalb um einige Prozentpunkte. Die Flächen unter den PR-Kurven sind aus den TensorBoard-Daten näherungsweise berechnet, TensorBoard zeigt sie selbst nicht an.




Ausgangslage
Das Testset hat 10 000 Bilder, davon sind genau 1000 Katzen. Das Netz gibt für jedes Bild eine Wahrscheinlichkeit aus, z.B. „dieses Bild ist zu 40 % eine Katze“.

Jetzt muss man festlegen, ab welchem Wert man sagt „das ist eine Katze“. Diese Grenze heißt Schwellwert.

Die zwei Fragen
Für einen festen Schwellwert stellt man zwei Fragen:

Precision: Wenn das Netz „Katze“ sagt, wie oft stimmt das? Ein Beispiel: Das Netz zeigt auf 100 Bilder und sagt „Katze“. Sind davon 65 wirklich Katzen, ist die Precision 65 %.
Recall: Von allen 1000 echten Katzen, wie viele findet das Netz? Findet es 230 davon, ist der Recall 23 %.
Ein Bild zum Merken ist ein Fischernetz:

Recall heißt: Wie viele der Fische im See habe ich gefangen?
Precision heißt: Wie viel von dem, was im Netz liegt, sind wirklich Fische und kein Müll?
Das Dilemma: Der Schwellwert verschiebt beides gegeneinander
Echte Zahlen für die Klasse Katze:

Schwellwert	sagt „Katze“	davon echt	Precision	Recall (von 1000)
10 % (großzügig)	3500	885	25 %	88 %
30 %	1218	540	44 %	54 %
50 %	356	230	65 %	23 %
70 % (streng)	58	44	76 %	4 %
Bei einem niedrigen Schwellwert sagt das Netz sehr oft „Katze“. Es findet fast alle Katzen (hoher Recall), aber drei von vier dieser Bilder sind gar keine Katzen (niedrige Precision).
Bei einem hohen Schwellwert sagt das Netz nur „Katze“, wenn es sehr sicher ist. Dann stimmt es meistens (hohe Precision), aber fast alle Katzen werden übersehen (niedriger Recall).
Die Kurve
Die PR-Kurve trägt einfach alle diese Paare aus Precision und Recall für sämtliche Schwellwerte als Linie ein:

x-Achse: Recall

y-Achse: Precision

Ein perfektes Netz hätte überall 100 % Precision bei jedem Recall. Die Kurve wäre dann eine Linie oben quer, die Fläche darunter wäre 1.

Ein schlechtes Netz hat eine Kurve, die schnell nach unten abfällt. Die Fläche darunter ist dann klein.

Die Fläche unter der Kurve fasst das in einer Zahl zusammen: Je näher an 1, desto besser trennt das Netz diese Klasse vom Rest.

Vergleich Auto und Katze
Zum Vergleich die gleiche Tabelle für Auto:

Schwellwert	Precision	Recall
30 %	80 %	77 %
50 %	89 %	64 %
Beim Auto bekommt man beides gleichzeitig hoch, bei der Katze nie. Das bedeuten die Flächen in der Tabelle: Auto 0,85, Katze 0,48. Das Netz ist sich bei Katzen oft unsicher und verwechselt sie mit Hunden, Hirschen usw.

Warum Softmax hier nötig ist
Für die Schwellwerte von 0 % bis 100 % braucht man echte Wahrscheinlichkeiten. Das Netz liefert aber nur rohe Werte (Logits), z.B. 3,7 oder −1,2. Softmax rechnet diese in Prozentwerte um, die zusammen 100 % ergeben.

Kurzantwort für den Professor
„Die PR-Kurve zeigt pro Klasse, wie sich Precision (wie oft stimmt die Vorhersage) und Recall (wie viele der echten Fälle werden gefunden) gegeneinander verschieben, wenn man den Entscheidungsschwellwert ändert. Je größer die Fläche darunter, desto besser trennt das Netz die Klasse vom Rest. Bei uns sind Fahrzeuge gut trennbar (z.B. car 0,85), Katzen am schlechtesten (0,48).“

Soll ich diese Erklärung mit der Tabelle auch in die Interpretationszelle im Notebook übernehmen?

# 4. TensorBoard

In [14]:
# TensorBoard-Erweiterung fuer Jupyter laden
%load_ext tensorboard
# startet TensorBoard direkt im Notebook (auskommentiert, falls es schon separat laeuft)
#%tensorboard --logdir runs/cifar10
# bereits laufendes TensorBoard auf Port 6006 im Notebook einbetten
from IPython.display import IFrame
IFrame('http://localhost:6006', width='100%', height=800)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
